<h3 style="color:#6FA8DC; font-weight:bold">04_Missing_Data_Random_Sample_Imputation</h3>

Handling Missing Data → Univariate Numerical Imputation

Reference basis: the provided CampusX numerical-imputation notebooks and `titanic_toy.csv`.

<h5 style="color:#78B89A; font-weight:bold;">1. What is univariate imputation? → one feature at a time</h5>

Univariate imputation fills missing values in a feature using information from **that same feature**.

Example:

`Age = [22, 38, NaN, 35, 28]`

The missing `Age` is filled using a rule based on the observed `Age` values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv('titanic_toy.csv')
df.head()

In [ ]:
df.isnull().mean() * 100

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

X_train.isnull().mean() * 100

<h5 style="color:#78B89A; font-weight:bold;">2. Why impute after train/test split? → avoid leakage</h5>

The value used to fill missing data must be **learned from the training set only**.

```text
X_train → learn imputation value
X_train → transform
X_test  → use the SAME learned value
```

Never calculate a mean/median using the complete dataset before splitting. That allows information from the test set to influence training.

<h5 style="color:#78B89A; font-weight:bold;">3. Method: Random Sample Imputation</h5>

<h5 style="color:#78B89A; font-weight:bold;">4. Idea → replace each missing value with a randomly sampled observed value</h5>

For a numerical feature, take a value from the **observed values of the same feature** and use it to fill a missing position.

Example:

`Age = [22, 38, NaN, 35, 28]`

A sampled observed age, such as `35`, can replace the missing value.

In [ ]:
rng = np.random.default_rng(42)

X_train['Age_random'] = X_train['Age']
missing_count = X_train['Age_random'].isna().sum()
observed_values = X_train['Age'].dropna().to_numpy()

X_train.loc[X_train['Age_random'].isna(), 'Age_random'] = rng.choice(
    observed_values, size=missing_count, replace=False
)

X_train[['Age','Age_random']].head(10)

<h5 style="color:#78B89A; font-weight:bold;">5. Why is it different from mean/median? → preserve observed-value distribution</h5>

Mean/median inserts the same value repeatedly. Random sample imputation inserts actual values sampled from the observed distribution.

That can preserve the shape and variance of the feature better than a single fixed statistic, although the result is random.

In [ ]:
print('Original variance:', X_train['Age'].var())
print('Random-imputed variance:', X_train['Age_random'].var())

fig, ax = plt.subplots(figsize=(8,4))
X_train['Age'].plot(kind='kde', ax=ax, label='Original')
X_train['Age_random'].plot(kind='kde', ax=ax, label='Random sample')
ax.legend(); ax.set_title('Age: Original vs Random Sample Imputation'); plt.show()

<h5 style="color:#78B89A; font-weight:bold;">6. Train/test rule → never sample test values</h5>

For ML, the values used to impute **both training and test data should come from the training distribution**.

```text
X_train observed values → sampling pool
                         ↓
              fill X_train missing
                         ↓
              fill X_test missing
```

Do not build the test sampling pool from test observations because that leaks information from the test set into preprocessing.

In [ ]:
rng = np.random.default_rng(42)

train_observed_age = X_train['Age'].dropna().to_numpy()

X_train_random = X_train[['Age','Fare']].copy()
X_test_random = X_test[['Age','Fare']].copy()

for col in ['Age', 'Fare']:
    train_values = X_train[col].dropna().to_numpy()
    n_train_missing = X_train[col].isna().sum()
    n_test_missing = X_test[col].isna().sum()
    
    if n_train_missing:
        X_train_random.loc[X_train_random[col].isna(), col] = rng.choice(train_values, n_train_missing, replace=True)
    if n_test_missing:
        X_test_random.loc[X_test_random[col].isna(), col] = rng.choice(train_values, n_test_missing, replace=True)

print(X_train_random.isnull().sum())
print(X_test_random.isnull().sum())

<h5 style="color:#78B89A; font-weight:bold;">7. Advantages / disadvantages</h5>

**Advantages:** uses real observed values, can preserve distribution better than constant imputation, keeps rows.

**Disadvantages:** random result, requires careful reproducibility, can duplicate observations, and is not a standard `SimpleImputer` strategy.

<h5 style="color:#78B89A; font-weight:bold;">8. When to use? → practical guidance</h5>

It can be useful when preserving the empirical distribution is important and the missing values are reasonably represented by the observed values.

For a production ML system, deterministic or carefully seeded transformer-based approaches are often easier to operationalize.

<h5 style="color:#78B89A; font-weight:bold;">9. Modern ML implementation → custom transformer / Pipeline</h5>

`SimpleImputer` does **not** have a built-in `random_sample` strategy. If you want this method inside a production pipeline, create or use a transformer that implements `fit()` and `transform()` and stores the training-data sampling pool.

The important production rule is: **fit on training data, then transform test/new data using the learned training distribution.**

<h5 style="color:#78B89A; font-weight:bold;">10. Final revision</h5>

```text
Observed values
      ↓
Sampling pool
      ↓
Random observed value
      ↓
Replace NaN

Train → learn sampling pool
Test / Production → sample only from training pool
```